# DuckPD Vector Search & Streaming Text Embeddings Walkthrough

This interactive notebook demonstrates how to use **DuckPD** for text embeddings and exact vector similarity search over remote and local Parquet datasets.

### Highlights
- **FastEmbed / ONNX runtime integration**: Seamlessly prepare and use quantized text embedding models locally on CPU.
- **Lazy remote streaming**: Scan Parquet datasets directly over HTTPS without downloading everything upfront.
- **In-engine text embedding**: Embed text columns lazily in bounded Arrow batches via `.embed_text()`.
- **Vector search API**: Query embedded datasets using `.vector.search_text()` with cosine distance, top-$k$ retrieval, and deterministic tie-breaking.

## 1. Imports and configuration

Define the remote source, a cache beside the notebook, the search query, and a revision-pinned embedding model. The cache path works whether the kernel starts in the repository root or in `demo/`.

In [1]:
from pathlib import Path
from time import perf_counter

import duckpd as pd

DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
EMBEDDED_DATA = DEMO_DIR / "nvidia-news-embedded.parquet"
QUERY = "AI chip demand and revenue growth"

MODEL = pd.embedding_model(
    "BAAI/bge-small-en-v1.5",
    revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    dimension=384,
)

print(f"DuckPD version: {pd.__version__}")
print(
    f"Embedding model: {MODEL.model} "
    f"(revision: {MODEL.revision[:12]}..., dimension: {MODEL.dimension})"
)
print(f"Embedded dataset: {EMBEDDED_DATA}")

DuckPD version: 0.1.4
Embedding model: BAAI/bge-small-en-v1.5 (revision: 5c38ec7c405e..., dimension: 384)
Embedded dataset: nvidia-news-embedded.parquet


## 2. Prepare the embedding model

Open a DuckPD session and prepare the text embedding model. Preparation initializes the local FastEmbed/ONNX backend and reports its available execution providers.

In [2]:
session = pd.connect()

preparation_started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - preparation_started

print(f"Backend: {prepared.backend} via {prepared.execution_providers}")
print(f"Model preparation time: {preparation_seconds:.3f}s")
print(f"Persisted model fingerprint: {MODEL.fingerprint}")

Backend: fastembed via ('CPUExecutionProvider',)
Model preparation time: 0.420s
Persisted model fingerprint: aaf9136d28a4dcfc7e6c2d3c81b90415fa40652b8dfe4a064fc3088f329ad352


## 3. Load or Build Embedded Dataset

If a pre-embedded Parquet file exists locally, we load it directly. Otherwise, DuckPD lazily scans the remote dataset from Hugging Face, filters for NVIDIA (`NVDA`) news articles, computes text embeddings across the `title` and `description` columns in batches, and persists the result to Parquet.

In [3]:
if EMBEDDED_DATA.exists():
    embedded = session.read_parquet(EMBEDDED_DATA)
    dataset_status = f"Loaded {EMBEDDED_DATA}"
else:
    build_started = perf_counter()
    news = session.read_parquet(DATA_URL)
    nvidia = news[news["symbol"] == "NVDA"]
    embedded = nvidia.embed_text(
        columns=["title", "description"],
        into="embedding",
        model=MODEL,
        batch_size=64,
        null_policy="empty",
    )
    embedded.write_parquet(EMBEDDED_DATA)
    build_seconds = perf_counter() - build_started
    dataset_status = (
        f"Created {EMBEDDED_DATA} from the remote archive "
        f"in {build_seconds:.3f} seconds"
    )
    embedded = session.read_parquet(EMBEDDED_DATA)

print(f"Embedding dataset: {dataset_status}")
print(f"Columns: {embedded.columns}")

Embedding text:   0%|          | 0/200 [00:00<?, ?documents/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Embedding dataset: Created nvidia-news-embedded.parquet from the remote archive in 36.313 seconds
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source', 'embedding')


## 4. Preview the source data

Inspect a few identifying columns with bounded materialization via `head()`. The large `embedding` vectors are intentionally omitted here; they remain available in the lazy frame for search.

In [4]:
preview = embedded[["symbol", "title", "publisher", "publish_date"]].head(5)
preview

,symbol,title,publisher,publish_date
0,NVDA,"Could $10,000 Invested in Nvidia Today Make Yo...",The Motley Fool,"Sep 6, 2026"
1,NVDA,Nvidia Is Near Its High While Its Biggest Chip...,The Motley Fool,"Sep 6, 2026"
2,NVDA,What to Invest in for the Next 5 Years: My Pre...,The Motley Fool,"Sep 6, 2026"
3,NVDA,Nvidia's Critical Shift Beyond GPUs Could Fuel...,The Motley Fool,"Sep 6, 2026"
4,NVDA,"Despite Revenue Skyrocketing More Than 100%, N...",The Motley Fool,"Sep 6, 2026"


## 5. Run a vector similarity search

Use `.vector.search_text()` to embed the query and retrieve the five nearest rows from the `embedding` column by cosine distance. The `title` tie-breaker keeps equally scored matches deterministic.

In [5]:
query_started = perf_counter()
matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    model=MODEL,
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]
result = matches.collect()
query_seconds = perf_counter() - query_started

print(f"Query: {QUERY!r}")
print(f"Query-to-response: {query_seconds:.3f} seconds")

Query: 'AI chip demand and revenue growth'
Query-to-response: 0.074 seconds


## 6. Inspect the results

Lower cosine distance means greater semantic similarity. Display the ranked matches, then close the DuckPD session.

In [6]:
display(result)
session.close()

,symbol,title,publisher,publish_date,_distance
0,NVDA,Better AI Infrastructure Stock: Nvidia vs. AMD,The Motley Fool,"Sep 3, 2026",0.235027
1,NVDA,Prediction: Nvidia Stock Will Double in Under ...,The Motley Fool,"Sep 6, 2026",0.235233
2,NVDA,You Could Buy Nvidia for Its 106% Revenue Grow...,The Motley Fool,"Aug 31, 2026",0.236833
3,NVDA,Upbeat AI Data Centers Demand Propels SK Hynix...,Zacks,"Sep 3, 2026",0.238607
4,NVDA,Broadcom vs. Nvidia: 1 Critical Metric Shows W...,The Motley Fool,"Sep 5, 2026",0.245503
